# 05 - Search Methods: Hybrid and Reranking

This notebook builds the third search method in two stages and scores it on the same 30 dev questions.

Two of the search methods we tried (bm25 and embeddings fail on different questions. Therefore, we want to check if a combination can cover more questions than either method alone.

The two stages:

- **Fusion** merges the ranked lists of BM25 and embeddings into one candidate list. Its job is coverage, meaning the evidence page should be somewhere among the candidates.
- **Reranking** re-sorts the top candidates with a cross-encoder, a model that reads the question and a chunk together and scores how well they match. It is too slow to run on all 33,998 chunks, which is why the cheap methods pre-select candidates first.

Where the settings come from:
- the chunks and BM25 settings from notebook 03
- the embedding model from notebook 04
- the fusion constant from Cormack et al. (2009)
- the reranker model (bge-reranker-v2-m3) from Kobeissi and Langlais (2026), who used it on this same dataset

This notebook runs on Google Colab.

In [1]:
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!pip install -q -U sentence-transformers qdrant-client
!pip install -q pymupdf bm25s
!python scripts/download_data.py

fatal: destination path 'financebench-rag' already exists and is not an empty directory.
/content/financebench-rag
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4_EARNINGS
[16/84] AMCOR_2023_10K
[17/84] AMD_2015_10K
[18/84] AMD_2022_10K
[19/84] AMERICANEXPRESS_2022_10K
[20/84] AMERICANWATERWORKS_2020_10K
[21/84] AMERICANWATERWORKS_2021_10K
[22/84] AMERICANWATERWORKS_2022_10K
[23/84] BESTBUY_2017_10K
[24/84] BESTBUY_2019_10K
[25/84] BESTBUY_2023_10K
[26/84] BESTBUY_2024Q2_10Q
[27/84] BLOCK_2016_10K
[28/84] BLOCK_2020_10K
[29/84] BOEING_2018_10K
[30/84] BOEING_2022_10K
[31/84] COCACOLA_2017_10K
[32/84] COCACOLA_2021_10K
[33/84] COCACOLA_2022_10K
[34/84] CORNING_2020_10K
[35/84] CORNI

In [3]:
import sys
sys.path.append("/content/financebench-rag/src")

## 1. Rebuild both searches

Both searches are rebuilt with the exact settings of notebooks 03 and 04. Afterwards, the dev scores of both are computed again and printed next to the saved results from notebooks 03 and 04, as a check that this notebook reproduces what the earlier ones measured.

In [4]:
import json
import time
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

from financebench_rag.data import REPO, extract_pages, gold_pages, load_doc_info, load_pages, load_questions, load_split
from financebench_rag.embedding import EmbeddingSearch
from financebench_rag.evaluate import evaluate_retrieval
from financebench_rag.retrieval import BM25Search, build_chunks, rrf_fuse

questions = load_questions()
dev_ids = set(load_split()["dev"])
dev = [q for q in questions if q["financebench_id"] in dev_ids]

extract_pages()
pages = load_pages()
chunks = build_chunks(pages, load_doc_info(), size=250, overlap=50, prefix=True)

bm25 = BM25Search(chunks)
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", model_kwargs={"dtype": "float16"})
model.max_seq_length = 512
start = time.time()
embeddings = EmbeddingSearch(chunks, model)
print(f"both searches ready in {(time.time() - start) / 60:.1f} min")

DEBUG:bm25s:Building index from IDs objects


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

... (7 outputs left out)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/532 [00:00<?, ?it/s]

/content/financebench-rag/src/financebench_rag/embedding.py:18: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert("chunks", points)


both searches ready in 25.4 min


In [5]:
def load_saved(name):
    return {fid: [tuple(page) for page in ranked] for fid, ranked in json.load(open(REPO / "results" / name)).items()}


dev_bm25 = {q["financebench_id"]: bm25.search(q["question"], k=10) for q in dev}
dev_emb = {q["financebench_id"]: embeddings.search(q["question"], k=10) for q in dev}

pd.DataFrame([
    {"ranking": "BM25 recomputed", **evaluate_retrieval(dev_bm25, dev)},
    {"ranking": "BM25 saved (notebook 03)", **evaluate_retrieval(load_saved("dev_retrieval_bm25.json"), dev)},
    {"ranking": "embeddings recomputed", **evaluate_retrieval(dev_emb, dev)},
    {"ranking": "embeddings saved (notebook 04)", **evaluate_retrieval(load_saved("dev_retrieval_embeddings.json"), dev)},
])

,ranking,recall@1,recall@5,mrr@10,doc_recall@5
0,BM25 recomputed,0.100,0.133,0.111,0.6
1,BM25 saved (notebook 03),0.100,0.133,0.111,0.6
2,embeddings recomputed,0.233,0.500,0.351,1.0
3,embeddings saved (notebook 04),0.233,0.500,0.351,1.0


## 2. Fusion

Reciprocal rank fusion (RRF) merges ranked lists without using their scores. A page's fused score is the sum of `1 / (k + rank)` over the lists it appears in, with `k = 60` as in Cormack et al. (2009). A page ranked high in both lists beats a page ranked high in only one. Both searches contribute their top 100 pages, so fusion has enough candidates to work with.

The fused list is the input to the reranker in section 3, so what matters most is whether the evidence page is among its candidates at all. The first table shows this as coverage, the share of questions whose evidence page is anywhere in the top 10, 20 or 50. The second table repeats the fused scores for k = 20, 60 and 100, to check that the choice of k does not drive the result.

In [6]:
deep_bm25 = {q["financebench_id"]: bm25.search(q["question"], k=100, n_chunks=2000) for q in dev}
deep_emb = {q["financebench_id"]: embeddings.search(q["question"], k=100, n_chunks=2000) for q in dev}
fused = {fid: rrf_fuse([deep_bm25[fid], deep_emb[fid]]) for fid in deep_bm25}


def coverage(results, depth):
    hits = sum(any(page in gold_pages(q) for page in results[q["financebench_id"]][:depth]) for q in dev)
    return round(hits / len(dev), 3)


pd.DataFrame([
    {"ranking": name, **{f"evidence in top {d}": coverage(ranking, d) for d in (10, 20, 50)}}
    for name, ranking in [("BM25", deep_bm25), ("embeddings", deep_emb), ("fused (RRF)", fused)]
])

,ranking,evidence in top 10,evidence in top 20,evidence in top 50
0,BM25,0.167,0.267,0.433
1,embeddings,0.700,0.867,0.967
2,fused (RRF),0.367,0.733,0.967


In [7]:
# Sensitivity of the fused ranking to the RRF constant k.
pd.DataFrame([
    {"rrf k": k, **evaluate_retrieval({fid: rrf_fuse([deep_bm25[fid], deep_emb[fid]], k=k) for fid in deep_bm25}, dev)}
    for k in (20, 60, 100)
])

,rrf k,recall@1,recall@5,mrr@10,doc_recall@5
0,20,0.1,0.233,0.191,0.867
1,60,0.1,0.233,0.161,0.767
2,100,0.1,0.233,0.162,0.767


Fusion with equal weights makes the candidate list worse than embeddings alone. In the top 20, the fused list contains the evidence page for 22 of 30 questions, against 26 of 30 for embeddings. In the top 10 the gap is larger, 11 against 21. BM25 is much weaker than embeddings on these questions, so giving both the same weight pushes good embedding pages down. At depth 50 both reach 29 of 30. The choice of k does not change this, since recall@5 is 0.233 for all three values.

## 3. Reranking

The reranker is bge-reranker-v2-m3, the cross-encoder that Kobeissi and Langlais (2026) used on this dataset. For each question it scores every chunk of every candidate page together with the question, and a page is then scored by its best chunk, the same rule both searches use. Re-sorting the candidates by these scores gives the final ranking.

The number of candidates passed to the reranker is tried at 20 and 50. In section 2, the fused top 50 contains the evidence page for 29 of 30 questions against 22 for the top 20, so 50 candidates give the reranker more to work with, , at a higher cost per question, since every candidate has to be scored.



In [8]:
# The embedding model is no longer needed once the ranked lists are computed,
# so it is removed from the GPU to make room for the reranker.
import gc

import torch

del model
embeddings.model = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory now: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

GPU memory now: 0.0 GB


In [9]:
from collections import defaultdict

from financebench_rag.rerank import Reranker

chunks_by_page = defaultdict(list)
for chunk in chunks:
    chunks_by_page[(chunk["doc_name"], chunk["page_num"])].append(chunk)

reranker = Reranker()
reranked = {}
for n in (20, 50):
    start = time.time()
    reranked[n] = {
        q["financebench_id"]: reranker.rerank(q["question"], fused[q["financebench_id"]][:n], chunks_by_page)
        for q in dev
    }
    print(f"reranked top {n}: {(time.time() - start) / len(dev):.1f} s per question")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

reranked top 20: 5.1 s per question
reranked top 50: 12.0 s per question


## 4. Scores on the dev questions

All search methods in one table, scored as in notebooks 03 and 04.

In [10]:
pd.DataFrame([
    {"method": "BM25 (notebook 03)", **evaluate_retrieval(dev_bm25, dev)},
    {"method": "embeddings (notebook 04)", **evaluate_retrieval(dev_emb, dev)},
    {"method": "fused (RRF, k=60)", **evaluate_retrieval(fused, dev)},
    {"method": "fused + rerank top 20", **evaluate_retrieval(reranked[20], dev)},
    {"method": "fused + rerank top 50", **evaluate_retrieval(reranked[50], dev)},
])

,method,recall@1,recall@5,mrr@10,doc_recall@5
0,BM25 (notebook 03),0.100,0.133,0.111,0.600
1,embeddings (notebook 04),0.233,0.500,0.351,1.000
2,"fused (RRF, k=60)",0.100,0.233,0.161,0.767
3,fused + rerank top 20,0.200,0.533,0.334,0.933
4,fused + rerank top 50,0.200,0.467,0.334,0.967


In [11]:
# Every dev question with the rank of its evidence page under the reranked ranking (top 20 candidates).
pd.set_option("display.max_colwidth", None)
chosen = reranked[20]  # the top 20 reranking, the best row of the table above
rows = []
for q in dev:
    rank = next((i + 1 for i, page in enumerate(chosen[q["financebench_id"]]) if page in gold_pages(q)), None)
    rows.append({"rank": rank, "question_type": q["question_type"], "question": q["question"][:130]})
pd.DataFrame(rows).sort_values("rank", na_position="last")

,rank,question_type,question
5,1.0,metrics-generated,What is the FY2017 operating cash flow ratio for Adobe? Operating cash flow ratio is defined as: cash from operations / total curr
12,1.0,metrics-generated,Answer the following question as if you are an equity research analyst and have lost internet connection so you do not have access
15,1.0,novel-generated,Was American Express able to retain card members during 2022?
23,1.0,novel-generated,"At the Pepsico AGM held on May 3, 2023, what was the outcome of the shareholder vote on the shareholder proposal for a congruency"
24,1.0,novel-generated,"As of FY2023Q1, why did Pepsico raise full year guidance for FY2023?"
26,1.0,novel-generated,"As of FY 2021, how much did Verizon expect to pay for its retirees in 2024?"
10,2.0,domain-relevant,"What are major acquisitions that AMCOR has done in FY2023, FY2022 and FY2021?"
6,2.0,domain-relevant,Does Adobe have an improving operating margin profile as of FY2022? If operating margin is not a useful metric for a company like
25,2.0,novel-generated,What drove the increase in Ulta Beauty's merchandise inventories balance at end of FY2023?
9,3.0,novel-generated,What was the key agenda of the AMCOR's 8k filing dated 1st July 2022?


- Reranking the fused top 20 raises recall@5 from 0.233 to 0.533, against 0.500 for embeddings alone. That is one question on 30, so on dev the two are even, and embeddings stay slightly ahead on recall@1 and mrr@10.
- Reranking the top 50 instead of the top 20 gives a recall@5 of 0.467 and takes more than twice as long, 12.0 s against 5.1 s per question (section 3).
- In the table above, the AMCOR 8-K and Ulta Beauty questions that only BM25 found in notebooks 03 and 04 are in the top 5, at ranks 3 and 2. So the combination does recover the questions it was meant to recover.

## 5. Save the results

The top 10 pages per dev question was saved in the same format as the other two result files, since files on Colab disappear when the session ends.

In [12]:
out_path = REPO / "results" / "dev_retrieval_hybrid.json"
out_path.write_text(json.dumps(chosen) + "\n")
print("wrote", out_path)

from google.colab import files
files.download(out_path)

wrote /content/financebench-rag/results/dev_retrieval_hybrid.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary of Results

- The rebuilt searches reproduce the notebook 03 and 04 scores exactly (section 1).
- Fusion with equal weights makes the candidate list worse than embeddings alone. The fused top 20 contains the evidence page for 22 of 30 questions, against 26 for embeddings, because BM25 is much weaker on these questions (section 2).
- Reranking the fused top 20 reaches a recall@5 of 0.533, against 0.500 for embeddings. That is one question on 30, so on dev the two are even, and embeddings stay slightly ahead on recall@1 and mrr@10 (section 4).
- Reranking the top 50 instead of the top 20 does not improve recall@5 and takes more than twice as long (sections 3 and 4).
- The AMCOR 8-K and Ulta Beauty questions that only BM25 found in notebooks 03 and 04 are recovered at ranks 3 and 2 (section 4).

Notebook 06 compares embeddings and fused + rerank top 20 (RRF with k=60 over the top 100 pages of both methods) on the 120 test questions. Since fusion weakened the candidate list, it also tests reranking the embedding top 20 without fusion.